In [36]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pandas as pd
from abc import ABC, abstractmethod
import numpy as np


In [37]:
class Layer(ABC):
    @abstractmethod
    def forward(self, X):
        pass
    
    @abstractmethod
    def backward(self, grad_output, learning_rate):
        pass

class Dense(Layer):
    def __init__(self, input_size, output_size, activation=None):
        self.input_size = input_size
        self.output_size = output_size
        self.activation = activation
        
        self.W = np.random.randn(input_size, output_size) * np.sqrt(2 / input_size)
        self.b = np.zeros((1, output_size))
        
        self.X = None
        self.Z = None
    
    def forward(self, X):
        self.X = X
        self.Z = X @ self.W + self.b
        
        if self.activation:
            return self.activation.forward(self.Z)
        return self.Z
    
    def backward(self, grad_output, learning_rate):
        if self.activation:
            grad = self.activation.backward(self.Z, grad_output)
        else:
            grad = grad_output
        
        if grad.ndim == 1:
            grad = grad.reshape(-1, 1)
        
        m = self.X.shape[0]
        dW = (self.X.T @ grad) / m
        db = np.mean(grad, axis=0, keepdims=True)
        
        grad_input = grad @ self.W.T
        
        self.W -= learning_rate * dW
        self.b -= learning_rate * db
        
        return grad_input

class Activation(ABC):
    @abstractmethod
    def forward(self, Z):
        pass
    
    @abstractmethod
    def backward(self, Z, grad_output):
        pass

class ReLU(Activation):
    def forward(self, Z):
        return np.maximum(0, Z)
    
    def backward(self, Z, grad_output):
        return grad_output * (Z > 0).astype(float)

class Sigmoid(Activation):
    def forward(self, Z):
        return 1 / (1 + np.exp(-np.clip(Z, -500, 500)))
    
    def backward(self, Z, grad_output):
        A = self.forward(Z)
        return grad_output * A * (1 - A)

class NeuralNetwork:
    def __init__(self):
        self.layers = []
    
    def add(self, layer):
        self.layers.append(layer)
        return self
    
    def forward(self, X):
        for layer in self.layers:
            X = layer.forward(X)
        return X
    
    def backward(self, grad_output, learning_rate):
        for layer in reversed(self.layers):
            grad_output = layer.backward(grad_output, learning_rate)
        return grad_output
    
    def fit(self, X, y, epochs=1000, learning_rate=0.01, batch_size=None, verbose=True):
        if y.ndim == 1:
            y = y.reshape(-1, 1)
        
        n_samples = X.shape[0]
        
        for epoch in range(epochs):
            indices = np.random.permutation(n_samples)
            X_shuffled = X[indices]
            y_shuffled = y[indices]
            
            if batch_size:
                for i in range(0, n_samples, batch_size):
                    X_batch = X_shuffled[i:i+batch_size]
                    y_batch = y_shuffled[i:i+batch_size]
                    self._train_batch(X_batch, y_batch, learning_rate)
            else:
                self._train_batch(X_shuffled, y_shuffled, learning_rate)
            
            if verbose and epoch % 100 == 0:
                y_pred = self.forward(X)
                loss = self._compute_loss(y, y_pred)
                print(f"Epoch {epoch:4d} | Loss: {loss:.6f}")
            elif epoch == epochs - 1:
                y_pred = self.forward(X)
                loss = self._compute_loss(y, y_pred)
                if verbose:
                    print(f"Epoch {epoch:4d} | Loss: {loss:.6f}")
        
        return self
    
    def _train_batch(self, X_batch, y_batch, learning_rate):
        y_pred = self.forward(X_batch)
        grad_output = self._loss_gradient(y_batch, y_pred)
        self.backward(grad_output, learning_rate)
    
    def _compute_loss(self, y_true, y_pred):
        eps = 1e-8
        return -np.mean(y_true * np.log(y_pred + eps) + (1 - y_true) * np.log(1 - y_pred + eps))
    
    def _loss_gradient(self, y_true, y_pred):
        return y_pred - y_true
    
    def predict(self, X):
        return self.forward(X)
    
    def predict_class(self, X, threshold=0.5):
        return (self.predict(X) >= threshold).astype(int)
    
    def score(self, X, y):
        if y.ndim == 1:
            y = y.reshape(-1, 1)
        y_pred = self.predict_class(X)
        return np.mean(y_pred == y)

In [38]:

data = pd.read_csv('train_data.csv')
X = data.drop(['PassengerId', 'Survived'], axis=1).values
y = data['Survived'].values

print(f"X shape: {X.shape}")  

scaler = StandardScaler()
X = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = NeuralNetwork()

model.add(Dense(X.shape[1], 32, activation=ReLU()))
model.add(Dense(32, 64, activation=ReLU()))
model.add(Dense(64, 32, activation=ReLU()))
model.add(Dense(32, 16, activation=ReLU()))
model.add(Dense(16, 1, activation=Sigmoid()))

model.fit(X_train, y_train, epochs=1000, learning_rate=0.01, batch_size=16, verbose=True)

print(f"Точность: {model.score(X_test, y_test):.4f}")

X shape: (792, 15)
Epoch    0 | Loss: 0.771404
Epoch  100 | Loss: 0.366165
Epoch  200 | Loss: 0.345347
Epoch  300 | Loss: 0.333723
Epoch  400 | Loss: 0.322088
Epoch  500 | Loss: 0.314204
Epoch  600 | Loss: 0.311908
Epoch  700 | Loss: 0.306957
Epoch  800 | Loss: 0.310701
Epoch  900 | Loss: 0.296918
Epoch  999 | Loss: 0.301418
Точность: 0.8302
